# Phase 3 Experiments - Phase A: Baselines + Prune Finetuned

This notebook runs:
- Run 1.1-1.4: All baseline evaluations (T1, T2, FS1, FS2)
- Run 3.1-3.2: Pruning on finetuned students (FS1, FS2)

**Estimated Time**: 3-4 hours

**Dependencies**: None (can run immediately)

**Total Runs**: 6

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes
!pip install -q iterative-stratification scikit-learn pandas numpy tqdm

In [ ]:
# Setup logging
import sys
from datetime import datetime

class Logger:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w")
    
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()
    
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger("/kaggle/working/experiment_log.txt")
print(f"Experiment started at: {datetime.now()}")
print("Phase A: Baselines + Prune Finetuned Students")

In [ ]:
# Clone repository
!git clone https://github.com/Saif-Siddique/phase_3_final.git
%cd phase_3_final
!git checkout phase3-comprehensive-experiments
!git log -1 --oneline

In [ ]:
# Verify dataset
import os
DATASET_PATH = "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv"

if os.path.exists(DATASET_PATH):
    import pandas as pd
    df = pd.read_csv(DATASET_PATH)
    print(f"Dataset loaded: {len(df)} samples")
    print(f"Columns: {df.columns.tolist()}")
else:
    print("ERROR: Dataset not found! Please add the dataset to your Kaggle notebook.")

---
## Scenario 1: Baselines (4 runs)
---

### Run 1.1: Teacher T1 - XLM-RoBERTa Baseline

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-xlm-roberta-base" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/T1_baseline

### Run 1.2: Teacher T2 - BanglaBERT Baseline

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-sagor-bangla-bert-base" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/T2_baseline

### Run 1.3: Finetuned Student FS1 - SahajBERT Baseline

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/FS1_baseline

### Run 1.4: Finetuned Student FS2 - BanglaBERT-small Baseline

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline baseline \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-csebuetnlp-banglabert_small" \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario1/FS2_baseline

---
## Scenario 3: Prune Finetuned Students (2 runs)
---

### Run 3.1: Prune Finetuned Student FS1 (SahajBERT-ft) with Magnitude

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-neuropark-sahajBERT" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/FS1_magnitude

### Run 3.2: Prune Finetuned Student FS2 (BanglaBERT-small-ft) with Magnitude

In [ ]:
!python main.py \
    --dataset_path "/kaggle/input/bangla-cyberbully-dataset/1_Multilablel_Cyberbully_Data.csv" \
    --author_name "Saif-Siddique" \
    --pipeline prune_only \
    --teacher_checkpoint "Saif-Siddique/bangla-cyberbully-csebuetnlp-banglabert_small" \
    --prune_method magnitude \
    --prune_sparsity 0.5 \
    --fine_tune_after_prune \
    --fine_tune_epochs 3 \
    --use_original_folds \
    --eval_fold 3 \
    --output_dir ./results/scenario3/FS2_magnitude

---
## Final Status Check & Save Results
---

In [ ]:
import os
import json
from datetime import datetime

print(f"\n{'='*60}")
print(f"EXPERIMENT COMPLETION STATUS - {datetime.now()}")
print(f"{'='*60}\n")

experiments = [
    # Scenario 1: Baselines
    ("1.1 T1 Baseline (XLM-RoBERTa)", "./results/scenario1/T1_baseline"),
    ("1.2 T2 Baseline (BanglaBERT)", "./results/scenario1/T2_baseline"),
    ("1.3 FS1 Baseline (SahajBERT-ft)", "./results/scenario1/FS1_baseline"),
    ("1.4 FS2 Baseline (BanglaBERT-small-ft)", "./results/scenario1/FS2_baseline"),
    # Scenario 3: Prune Finetuned
    ("3.1 FS1 + Magnitude Pruning", "./results/scenario3/FS1_magnitude"),
    ("3.2 FS2 + Magnitude Pruning", "./results/scenario3/FS2_magnitude"),
]

success_count = 0
results_summary = []

for name, output_dir in experiments:
    json_path = os.path.join(output_dir, "results_final.json")
    
    if os.path.exists(json_path):
        with open(json_path) as f:
            data = json.load(f)
        f1_macro = data.get('f1_macro', 'N/A')
        print(f"[SUCCESS] {name}: F1 Macro = {f1_macro:.4f}")
        results_summary.append({"experiment": name, "f1_macro": f1_macro, "status": "SUCCESS"})
        success_count += 1
    else:
        print(f"[FAILED] {name}: No results found")
        results_summary.append({"experiment": name, "status": "FAILED"})

print(f"\n{'='*60}")
print(f"Completed: {success_count}/{len(experiments)} experiments")
print(f"{'='*60}")

In [ ]:
# Copy results to Kaggle output
!cp -r ./results /kaggle/working/
!ls -la /kaggle/working/results/

In [ ]:
# Run aggregation script
!python aggregate_results.py --results_dir ./results --output /kaggle/working/phase_a_summary.csv --format all

In [ ]:
print(f"\nPhase A completed at: {datetime.now()}")
print("\nNOTE: Phase A is independent. You can run Phase B (KD) in parallel.")
print("      Phase C+D DEPENDS on Phase B completing and uploading KD models!")